In [1]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader

class AudioDataset(Dataset):
    def __init__(self, tensors, labels):
        self.tensors = tensors  # (N, T, F) -> N örnek sayısı, T zaman adımı, F özellik boyutu
        self.labels = labels  # Speaker ID veya sınıf etiketi

    def __len__(self):
        return len(self.tensors)

    def __getitem__(self, idx):
        return self.tensors[idx], self.labels[idx]

tensors = torch.load('audio_tensors.pt')
df = pd.read_csv("processed.tsv", sep="\t")
labels = df["speaker_id"].values
dataset = AudioDataset(tensors, labels)
dataloader = DataLoader(dataset, batch_size=13, shuffle=True)

/tmp/ipykernel_3831/44005724.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tensors = torch.load('audio_tensors.pt')


In [2]:
import torch.nn as nn

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(RNNModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # Başlangıç gizli durumu
        out, _ = self.gru(x, h0)  # (batch, seq_len, hidden_size)
        out = self.fc(out[:, -1, :])  # Son zaman adımını al ve sınıflandır
        return out

# Modeli cihaza yükle (GPU varsa kullan, yoksa CPU'yu kullan)
device = torch.device("cuda")

# Modeli oluştur ve cihaza taşı
model = RNNModel(input_size=29, hidden_size=128, output_size=64).to(device)


In [9]:
import torch.optim as optim
# Kayıp fonksiyonu ve optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Eğitim parametreleri
num_epochs = 20  # Kaç epoch boyunca eğitileceği
# Eğitim döngüsü
for epoch in range(num_epochs):
    total_loss = 0.0
    correct = 0
    total = 0

    model.train()  # Modeli eğitim moduna al

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)  # Verileri cihaza taşı

        # Verileri uygun şekle getir (batch_size, seq_len, input_size)
        inputs = inputs.view(inputs.size(0), 313, 29)  # (batch, seq_length, feature_size)

        optimizer.zero_grad()  # Gradients'i sıfırla
        outputs = model(inputs)  # Modeli ileri besle

        loss = criterion(outputs, labels)  # Kayıp hesapla
        loss.backward()  # Geri yayılım (backpropagation)
        optimizer.step()  # Ağırlıkları güncelle

        total_loss += loss.item()

        # Doğruluk hesapla
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

print("Eğitim tamamlandı!")


Epoch [1/20], Loss: 0.6868, Accuracy: 81.16%
Epoch [2/20], Loss: 0.7031, Accuracy: 80.43%
Epoch [3/20], Loss: 0.6022, Accuracy: 88.41%
Epoch [4/20], Loss: 0.5493, Accuracy: 86.96%
Epoch [5/20], Loss: 0.6220, Accuracy: 84.06%
Epoch [6/20], Loss: 0.5600, Accuracy: 86.96%
Epoch [7/20], Loss: 0.6616, Accuracy: 83.33%
Epoch [8/20], Loss: 0.8817, Accuracy: 76.81%
Epoch [9/20], Loss: 0.9890, Accuracy: 76.81%
Epoch [10/20], Loss: 0.7672, Accuracy: 79.71%
Epoch [11/20], Loss: 0.7114, Accuracy: 82.61%
Epoch [12/20], Loss: 0.6592, Accuracy: 87.68%
Epoch [13/20], Loss: 0.5917, Accuracy: 84.78%
Epoch [14/20], Loss: 0.5684, Accuracy: 87.68%
Epoch [15/20], Loss: 0.5906, Accuracy: 87.68%
Epoch [16/20], Loss: 0.5653, Accuracy: 89.13%
Epoch [17/20], Loss: 0.6262, Accuracy: 84.78%
Epoch [18/20], Loss: 0.6182, Accuracy: 85.51%
Epoch [19/20], Loss: 0.5870, Accuracy: 86.96%
Epoch [20/20], Loss: 0.6046, Accuracy: 84.78%
Eğitim tamamlandı!


In [5]:
torch.save(model.state_dict(), "speaker_identification_model.pth")
print("Model kaydedildi!")

Model kaydedildi!
